# ViT from scratch : briques de base

| | |
|---|---|
| **Architecture** | Vision Transformer (ViT) |
| **Sujet** | Découpage d'une image en patches et encodage de leur relation spatiale |
| **Papier** | [Dosovitskiy et al. 2020 (*An Image is Worth 16x16 Words*)](https://arxiv.org/abs/2010.11929) |
| **Composants couverts** | Patch embedding · class token · positional embedding · attention multi-têtes · bloc Transformer |
| **Langage** | Python / PyTorch |

---

Ce notebook suit la même logique que le papier : on part d'une image, on la découpe en patches, on ajoute une information de position, puis on fait circuler l'information à travers l'attention multi-têtes pour produire une représentation utile à la classification.

L'objectif n'est pas seulement de montrer le code, mais de relier chaque bloc à son rôle conceptuel : pourquoi les patches, pourquoi le class token, pourquoi un embedding positionnel, et comment tout cela s'articule dans un petit Vision Transformer.

## Notation image et nombre de patches

On note l'entrée image:

$$
x \in \mathbb{R}^{B \times C \times H \times W}
$$

avec:

- $B$: taille de batch,
- $C$: nombre de canaux,
- $H$: hauteur,
- $W$: largeur.

Si l'image est découpée en patches carrés de taille $P \times P$, alors le nombre de patches par image vaut:

$$
N = \frac{HW}{P^2}
$$

pourvu que $H$ et $W$ soient divisibles par $P$.

In [ ]:
import math

import torch
from torch import nn

from vit_from_scratch import (
    MLP,
    MultiHeadSelfAttention,
    PatchEmbedding,
    TransformerEncoderBlock,
    ViTConfig,
    VisionTransformer,
)

torch.manual_seed(7)

In [ ]:
B, C, H, W = 2, 3, 32, 32
patch_size = 8

images = torch.linspace(0.0, 1.0, steps=B * C * H * W, dtype=torch.float32)
images = images.reshape(B, C, H, W)

num_patches = (H * W) // (patch_size ** 2)

print('images shape:', tuple(images.shape))
print('nombre de patches par image:', num_patches)

## De l'image aux patch tokens

Dans ViT, chaque patch est aplati puis projeté dans un espace latent de dimension $D$. Le papier résume cette étape par:

$$
z_0 = [x_{class}; x_p^1 E; \ldots; x_p^N E] + E_{pos}
$$

où:

- $x_p^i$ désigne le $i$-ème patch aplati, de dimension $C \cdot P^2$,
- $E \in \mathbb{R}^{(C \cdot P^2) \times D}$ est la matrice de projection linéaire des patches,
- $x_{class}$ est un token appris prépendé à la séquence,
- $E_{pos} \in \mathbb{R}^{(N+1) \times D}$ est l'embedding positionnel appris.

La séquence résultante contient $N + 1$ tokens de dimension $D$.

In [ ]:
embed_dim = 64

config = ViTConfig(
    image_size=H,
    patch_size=patch_size,
    in_channels=C,
    num_classes=10,
    embed_dim=embed_dim,
    depth=2,
    num_heads=4,
    mlp_ratio=4.0,
    dropout=0.0,
    attention_dropout=0.0,
)

patch_embed = PatchEmbedding(config)
patch_tokens = patch_embed(images)

print('patch tokens shape:', tuple(patch_tokens.shape))
print('attendu = (B, N, D):', (B, num_patches, embed_dim))

In [ ]:
flat_patch_dim = C * patch_size * patch_size
unfold = nn.Unfold(kernel_size=patch_size, stride=patch_size)
raw_patches = unfold(images).transpose(1, 2)

print('raw patches shape:', tuple(raw_patches.shape))
print('dimension d\'un patch aplati:', flat_patch_dim)

## Ajout du class token et des positions

Le `class token` est un vecteur appris qui agrège l'information de tous les patches via l'attention. Sa représentation finale sert d'entrée à la tête de classification.

On le prépend à la séquence avant d'ajouter les positions, pour que chaque token (y compris celui de classe) reçoive un indice positionnel.

> **Analogie, le rapporteur de classe**  
> Imaginez que chaque patch est un témoin qui a observé une zone précise de l'image. Le *class token* est un rapporteur qui n'a rien vu directement : il assiste à l'audience, écoute tous les témoins grâce au mécanisme d'attention, et rédige à la fin un compte-rendu synthétique. C'est ce compte-rendu (la représentation finale du class token) qui est transmis à la tête de classification.

In [ ]:
cls_token = torch.zeros(B, 1, embed_dim)
pos_embed = torch.zeros(1, num_patches + 1, embed_dim)

z0 = torch.cat([cls_token, patch_tokens], dim=1) + pos_embed

print('z0 shape:', tuple(z0.shape))
print('longueur de séquence attendue:', num_patches + 1)

## LayerNorm : stabiliser les activations

Dans un bloc pre-norm, on applique une normalisation avant l'attention ou le MLP. Pour un token donné, `LayerNorm` recentre et re-échelle les dimensions de caractéristiques, ce qui régularise les distributions d'une couche à l'autre.

Formellement, pour un vecteur $x \in \mathbb{R}^D$ :

$$
\mathrm{LN}(x) = \gamma \odot \frac{x - \mu}{\sigma + \epsilon} + \beta
$$

où $\mu$ et $\sigma$ sont la moyenne et l'écart-type calculés **sur les $D$ dimensions** du token (et non sur le batch), et $\gamma, \beta$ sont des paramètres appris.

On peut comparer la moyenne et l'écart-type avant et après normalisation.

> **Pourquoi normaliser avant l'attention ?**  
> Dans le bloc pre-norm, la `LayerNorm` est appliquée *avant* chaque sous-bloc (attention ou MLP). L'intuition est double. Premièrement, si les activations ont des amplitudes très variables d'un token à l'autre, les produits scalaires $QK^T$ explosent, le softmax sature, et les gradients disparaissent. Deuxièmement, normaliser stabilise les distributions de couche en couche, ce qui accélère la convergence et rend l'entraînement des modèles profonds beaucoup plus robuste.

In [ ]:
layer_norm = nn.LayerNorm(embed_dim)
normalized_tokens = layer_norm(z0)

mean_before = z0.mean(dim=-1)
std_before = z0.std(dim=-1, unbiased=False)
mean_after = normalized_tokens.mean(dim=-1)
std_after = normalized_tokens.std(dim=-1, unbiased=False)

print('moyenne avant LN, premier batch:', mean_before[0, :4])
print('std avant LN, premier batch:', std_before[0, :4])
print('moyenne après LN, premier batch:', mean_after[0, :4])
print('std après LN, premier batch:', std_after[0, :4])

## Self-attention

La self-attention projette les tokens en trois représentations : $Q$ (queries), $K$ (keys) et $V$ (values). La formule de base est:

$$
\mathrm{Attention}(Q, K, V) = \mathrm{softmax}\left(\frac{QK^T}{\sqrt{d_k}}\right)V
$$

$QK^T$ mesure la compatibilité entre tokens, la division par $\sqrt{d_k}$ stabilise l'échelle, et le softmax convertit les scores en poids d'attention. Sans cette normalisation, quand $d_k$ est grand, les produits scalaires deviennent trop grands et le softmax sature vers des distributions quasi-binaires, ce qui bloque les gradients.

In [ ]:
num_heads = config.num_heads
head_dim = embed_dim // num_heads

qkv = nn.Linear(embed_dim, 3 * embed_dim, bias=False)
qkv_tokens = qkv(normalized_tokens)
qkv_tokens = qkv_tokens.reshape(B, num_patches + 1, 3, num_heads, head_dim) # (B, N, 3, num_heads, head_dim)
qkv_tokens = qkv_tokens.permute(2, 0, 3, 1, 4) # (3, B, num_heads, N, head_dim)
Q, K, V = qkv_tokens[0], qkv_tokens[1], qkv_tokens[2] # Q, K, V shape: (B, num_heads, N, head_dim)

scores = (Q @ K.transpose(-2, -1)) / math.sqrt(head_dim)
attention_weights = scores.softmax(dim=-1)
context = attention_weights @ V

print('Q shape:', tuple(Q.shape))
print('scores shape:', tuple(scores.shape))
print('attention weights shape:', tuple(attention_weights.shape))
print('context shape:', tuple(context.shape))

## Multi-head attention

L'idée multi-têtes consiste à apprendre plusieurs sous-espaces d'attention en parallèle:

$$
\mathrm{MHA}(x) = \mathrm{Concat}(\mathrm{head}_1, \ldots, \mathrm{head}_h)W^O
$$

avec $\mathrm{head}_i = \mathrm{Attention}(xW_i^Q,\, xW_i^K,\, xW_i^V)$.

Chaque tête opère sur un sous-espace de dimension $d_k = D / h$, ce qui maintient le coût de calcul total comparable à une attention à tête unique de dimension $D$. Les sorties sont concaténées, puis reprojetées dans la dimension `D` via $W^O \in \mathbb{R}^{D \times D}$.

> **Analogie, les lunettes spécialisées**  
> Une seule tête d'attention ne peut regarder qu'un seul type de relation entre patches. L'attention multi-têtes permet à chaque tête d'observer l'image avec des «lunettes» différentes : une tête peut se spécialiser sur les contours, une autre sur les régions de même couleur, une autre sur les relations structurelles entre zones éloignées. En concaténant leurs résultats, le modèle obtient une description bien plus riche qu'avec un seul point de vue.

In [ ]:
mha = MultiHeadSelfAttention(
    embed_dim=config.embed_dim,
    num_heads=config.num_heads,
    dropout=config.dropout,
    attention_dropout=config.attention_dropout,
)
mha_output = mha(normalized_tokens)

print('sortie MHA shape:', tuple(mha_output.shape))

## Bloc encodeur Transformer pre-norm

Le bloc encodeur de ViT utilise des connexions résiduelles et une normalisation avant chaque sous-bloc:

$$
y = x + \mathrm{MSA}(\mathrm{LN}(x))
$$

$$
z = y + \mathrm{MLP}(\mathrm{LN}(y))
$$

Le choix pre-norm (normaliser avant chaque sous-bloc) stabilise l'entraînement des modèles profonds, contrairement au post-norm original qui peut diverger. Les connexions résiduelles garantissent que le gradient peut circuler directement depuis la sortie jusqu'à l'entrée du bloc, sans être atténué par les opérations non-linéaires.

In [ ]:
mlp = MLP(
    embed_dim=config.embed_dim,
    hidden_dim=config.mlp_hidden_dim,
    dropout=config.dropout,
)
mlp_output = mlp(normalized_tokens)

encoder_block = TransformerEncoderBlock(
    embed_dim=config.embed_dim,
    num_heads=config.num_heads,
    mlp_hidden_dim=config.mlp_hidden_dim,
    dropout=config.dropout,
    attention_dropout=config.attention_dropout,
)
block_output = encoder_block(z0)

print('sortie MLP shape:', tuple(mlp_output.shape))
print('sortie bloc encodeur shape:', tuple(block_output.shape))

## Fermer la boucle avec le modèle ViT

Le modèle complet empile plusieurs blocs encodeurs sur la séquence $z_0$. La méthode `encode_tokens(images)` renvoie la représentation tokenisée et encodée avant la tête de classification.

Le `class token` final (premier token de la séquence) est envoyé dans une couche linéaire pour prédire la classe.

In [ ]:
vit = VisionTransformer(config)
encoded_tokens = vit.encode_tokens(images)
cls_representation = encoded_tokens[:, 0]

print('encoded tokens shape:', tuple(encoded_tokens.shape))
print('class token final shape:', tuple(cls_representation.shape))

## Conclusion

### Ce que nous avons construit

Ce notebook a suivi la chaîne complète du ViT, des pixels à la prédiction de classe :

| Brique | Rôle |
|---|---|
| **Découpage en patches** | Transformer $x \in \mathbb{R}^{B \times C \times H \times W}$ en $N = HW/P^2$ tokens de dimension $C \cdot P^2$ |
| **Projection linéaire $E$** | Projeter chaque patch aplati vers l'espace latent de dimension $D$ |
| **Class token + positions** | Ajouter le vecteur rapporteur et encoder l'ordre spatial |
| **LayerNorm** | Stabiliser les activations avant chaque sous-bloc |
| **Self-attention** | Calculer $\mathrm{softmax}(QK^T / \sqrt{d_k})V$ pour relier tous les patches |
| **Multi-head attention** | Apprendre plusieurs relations en parallèle, chaque tête avec ses propres lunettes |
| **Bloc encodeur pre-norm** | Combiner MHA et MLP avec connexions résiduelles : $z = y + \mathrm{MLP}(\mathrm{LN}(y))$ |
| **VisionTransformer** | Empiler $L$ blocs, extraire le class token, classifier |

### Pour aller plus loin

Les notebooks suivants de cette série explorent les points que nous avons laissés de côté ici :

- **`embedding_methods.ipynb`**, stratégies d'embedding positionnel : appris vs. sinusoïdal vs. RoPE vs. 2D, et pourquoi le choix influence les capacités d'extrapolation hors-distribution.
- **`training_methods.ipynb`**, objectifs d'entraînement alternatifs : supervision directe, reconstruction masquée (MAE), auto-supervision teacher-student (DINO).

### L'écosystème ViT

Le ViT original a donné lieu à plusieurs variantes, chacune ciblant une limitation différente :

- **DeiT** (Touvron et al., 2021), entraînement efficace sur ImageNet sans pré-entraînement massif, grâce à la distillation et à des augmentations agressives.
- **BEiT** (Bao et al., 2022), pré-entraînement par prédiction de tokens visuels masqués, analogue à BERT en NLP.
- **MAE** (He et al., 2022), auto-encodeur masqué : reconstruire 75 % de l'image à partir de 25 % de patches visibles.
- **DINO / DINOv2** (Caron et al., 2021 ; Oquab et al., 2023), auto-supervision sans labels, dont les représentations révèlent naturellement la segmentation sémantique.

La brique commune à toutes ces variantes est exactement ce que vous venez de construire.